# Meta Thinker: Meta-Learning Reasoning Strategies

This notebook demonstrates the **Meta Thinker** approach - a system that learns to select appropriate reasoning strategies for different problem types, and even invents new strategies when existing ones fail.

## Core Concept

Instead of fixing a single reasoning strategy (like Chain-of-Thought), Meta Thinker:
1. **Analyzes** the problem type
2. **Selects** the most appropriate reasoning strategy from known strategies (CoT, ToT, AoT)
3. **Invents** new reasoning strategies when existing ones are insufficient
4. **Learns** from examples across multiple datasets

## What You'll Learn

1. How to define and use multiple reasoning strategies
2. How to create a meta-reasoning prompt that selects strategies
3. How to test the system on various reasoning tasks
4. How to compare meta-learning with traditional approaches

## Prerequisites

Make sure you have:
- Completed notebook `00_setup_and_data.ipynb`
- Set up API keys for GPT-4 or DeepSeek
- Installed utils modules from the project

## 1. Setup and Imports

In [ ]:
# Import utility functions
from utils.utils import *
from utils.agent import *
import os
import json
import numpy as np
from typing import Dict, List

print("✓ Imports successful")

## 2. Initialize LLM Client

We'll use GPT-4 or DeepSeek for meta-reasoning experiments.

In [ ]:
# Choose your model: 'gpt' or 'deepseek'
MODEL_TYPE = 'gpt'  # Change to 'deepseek' if you prefer

# Initialize the client
client = initialize_pipeline(MODEL_TYPE)

print(f"✓ Initialized {MODEL_TYPE} client")

## 3. Define the Meta-Reasoning System Prompt

This is the core of Meta Thinker - a system prompt that teaches the model to:
- Understand different reasoning strategies
- Choose the appropriate strategy for each problem
- Invent new strategies when needed

In [ ]:
META_REASONING_PROMPT = """You are a meta-reasoning agent. Your goal is to solve reasoning problems efficiently and accurately.

You have access to several known reasoning strategies:

1. **Chain of Thought (CoT)**: "Let's think step by step."
   - Best for: Linear problems with clear sequential steps
   - Examples: Basic math, logical deduction

2. **Tree of Thought (ToT)**: "Imagine three experts solving the problem in parallel, sharing thoughts at each step, eliminating paths that fail."
   - Best for: Problems with multiple possible approaches
   - Examples: Puzzles, optimization problems

3. **Algorithm of Thought (AoT)**: "First perform a forward analysis of the problem, then backtrack from the goal to verify or revise your solution."
   - Best for: Complex problems requiring verification
   - Examples: Strategy problems, proofs

Your job is to:
1. Analyze the problem type and complexity
2. Choose the most appropriate existing strategy for the task
3. If you think none of the known strategies work optimally, invent a new reasoning strategy
4. Clearly label the strategy you use
5. Use step-by-step reasoning and provide a final answer

**Important**: Only invent a new reasoning style if the existing ones fail to solve the problem effectively.

Use the following format:

<strategy>
[Strategy name: CoT, ToT, AoT, or a new invented strategy if required]
</strategy>

<think>
[Your step-by-step reasoning using the chosen strategy]
</think>

<answer>
[Your final answer]
</answer>
"""

print("✓ Meta-reasoning system prompt defined")
print(f"\nPrompt length: {len(META_REASONING_PROMPT)} characters")

## 4. Test Meta-Reasoning on a Simple Problem

Let's start with the 24-point game to see how the model selects a strategy.

In [ ]:
# Define a test problem
test_problem = """Your task is to solve a 24-point arithmetic puzzle.

Use exactly the four numbers: 4 6 1 1
You may use the operations: +, −, ×, ÷
You must use all four numbers exactly once
Your goal is to combine them to get 24

Remember to:
- First choose the reasoning strategy
- Then solve step-by-step
- Output your answer in the required format
"""

# Create message with system prompt and user problem
messages = [
    {"role": "system", "content": META_REASONING_PROMPT},
    {"role": "user", "content": test_problem}
]

# Get response
response = pipeline_message(messages, client)

print("=" * 70)
print("META THINKER RESPONSE:")
print("=" * 70)
print(response)
print("=" * 70)

## 5. Load Training Examples from Multiple Datasets

To improve meta-learning, we provide examples from different reasoning domains:
- **GSM8K**: Math word problems
- **Game24**: Arithmetic puzzles
- **PIQA**: Physical commonsense
- **CommonsenseQA**: General commonsense reasoning

In [ ]:
# Dataset names
DATASET_NAMES = ['gsm8k', 'game24', 'piqa', 'commonsense']

# Load all datasets
datasets = {}
for name in DATASET_NAMES:
    file_path = f'./input_data/qa_dataset_{name}.json'
    try:
        with open(file_path, 'r') as file:
            datasets[name] = json.load(file)
        print(f"✓ Loaded {name}: {len(datasets[name])} examples")
    except FileNotFoundError:
        print(f"⚠ Warning: {file_path} not found")

print(f"\n✓ Total datasets loaded: {len(datasets)}")

## 6. Create Few-Shot Learning Prompt

We'll create a few-shot prompt by sampling examples from each dataset, showing:
- The question
- The optimal reasoning strategy used
- The solution

In [ ]:
def create_few_shot_prompt(datasets: Dict, n_examples: int = 3) -> List[Dict]:
    """
    Create a few-shot learning prompt from multiple datasets.
    
    Args:
        datasets: Dictionary of dataset_name -> data
        n_examples: Number of examples to sample from each dataset
    
    Returns:
        List of message dictionaries for the conversation
    """
    messages = []
    
    # Add system prompt
    messages.append({
        'role': 'system',
        'content': META_REASONING_PROMPT
    })
    
    # Add examples from each dataset
    for dataset_name, data in datasets.items():
        if len(data) == 0:
            continue
            
        # Sample random examples
        n_samples = min(n_examples, len(data))
        indices = np.random.randint(0, len(data), n_samples)
        
        # Introduce the dataset
        messages.append({
            'role': 'user',
            'content': (
                f"I will now show you {n_samples} examples from the {dataset_name} dataset. "
                f"Each example includes the optimal reasoning strategy that works best."
            )
        })
        
        # Add each example
        for idx in indices:
            example = data[idx]
            question = example['question']
            best_strategy = example['best']
            answer = example[best_strategy][0]
            
            messages.append({
                'role': 'system',
                'content': f"Example from {dataset_name}:\n\nQuestion: {question}"
            })
            
            messages.append({
                'role': 'user',
                'content': (
                    f"The optimal reasoning style for this is '{best_strategy}'. \n\n"
                    f"Solution:\n{answer}"
                )
            })
    
    return messages

# Create few-shot prompt
few_shot_messages = create_few_shot_prompt(datasets, n_examples=2)

print(f"✓ Created few-shot prompt with {len(few_shot_messages)} messages")
print(f"✓ Covering {len(datasets)} different reasoning domains")

## 7. Test Meta-Learning with Few-Shot Examples

Now let's test if the model can learn to select better strategies after seeing examples.

In [ ]:
# Define a challenging problem that requires creative reasoning
challenging_problem = """
Now let's try a harder version of the 24-point game.

Use FIVE numbers to compute 24: 2, 4, 5, 8, 6
- Use each number exactly once
- Use only +, -, *, / operators
- You can use parentheses

If existing thinking styles don't work well, feel free to invent a new reasoning approach.
"""

# Add the new problem to few-shot messages
test_messages = few_shot_messages.copy()
test_messages.append({
    'role': 'user',
    'content': challenging_problem
})

# Get response from meta-learning model
print("Testing Meta Thinker with few-shot learning...\n")
meta_response = pipeline_message(test_messages, client)

print("=" * 70)
print("META-LEARNING RESPONSE (WITH FEW-SHOT EXAMPLES):")
print("=" * 70)
print(meta_response)
print("=" * 70)

## 8. Compare with Traditional Reasoning Strategies

Let's compare the meta-learning approach with traditional fixed strategies (CoT, ToT, AoT).

In [ ]:
def test_with_fixed_strategy(client, problem: str, style: str, task: str = 'game24') -> str:
    """
    Test a problem using a fixed reasoning strategy.
    
    Args:
        client: LLM client
        problem: The problem to solve
        style: Reasoning style (CoT, ToT, AoT)
        task: Task type
    
    Returns:
        Model's response
    """
    response = get_response(
        client,
        style=style,
        encourage_text=True,
        input=problem,
        base_model=MODEL_TYPE,
        task=task
    )
    return response

# Test with each traditional strategy
traditional_strategies = ['CoT', 'AoT', 'ToT']
traditional_responses = {}

print("Testing traditional reasoning strategies...\n")

for style in traditional_strategies:
    print(f"Testing {style}...")
    response = test_with_fixed_strategy(client, challenging_problem, style)
    traditional_responses[style] = response
    print(f"✓ {style} complete\n")

print("✓ All traditional strategies tested")

## 9. Display Comparison Results

Compare all approaches side-by-side.

In [ ]:
print("=" * 80)
print("COMPARISON OF REASONING APPROACHES")
print("=" * 80)
print(f"\nProblem: {challenging_problem}\n")
print("=" * 80)

# Meta-learning response
print("\n[1] META THINKER (Adaptive Strategy Selection)")
print("-" * 80)
print(meta_response[:500] + "...\n" if len(meta_response) > 500 else meta_response + "\n")

# Traditional strategies
for i, (style, response) in enumerate(traditional_responses.items(), 2):
    print(f"\n[{i}] {style} (Fixed Strategy)")
    print("-" * 80)
    print(response[:500] + "...\n" if len(response) > 500 else response + "\n")

print("=" * 80)

## 10. Save Results for Analysis

Save all responses to a file for further analysis.

In [ ]:
def save_comparison_results(problem: str, meta_response: str, 
                           traditional_responses: Dict[str, str],
                           filename: str = 'meta_learning_comparison.txt'):
    """
    Save comparison results to a file.
    """
    output_path = f'./results/{filename}'
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write("=" * 80 + "\n")
        f.write("META THINKER: REASONING STRATEGY COMPARISON\n")
        f.write("=" * 80 + "\n\n")
        
        f.write("## Problem\n")
        f.write(problem + "\n\n")
        
        f.write("## Meta Thinker Response (Adaptive)\n")
        f.write(meta_response + "\n\n")
        f.write("-" * 80 + "\n\n")
        
        for style, response in traditional_responses.items():
            f.write(f"## {style} Response (Fixed)\n")
            f.write(response + "\n\n")
            f.write("-" * 80 + "\n\n")
    
    print(f"✓ Results saved to: {output_path}")

# Save results
save_comparison_results(
    problem=challenging_problem,
    meta_response=meta_response,
    traditional_responses=traditional_responses,
    filename='game24_5numbers_comparison.txt'
)

## 11. Analyze Strategy Selection Patterns

Extract which strategy the meta-learner chose and why.

In [ ]:
def extract_strategy(response: str) -> str:
    """
    Extract the strategy name from a meta-reasoning response.
    """
    if '<strategy>' in response and '</strategy>' in response:
        start = response.find('<strategy>') + len('<strategy>')
        end = response.find('</strategy>')
        strategy = response[start:end].strip()
        return strategy
    return "Unknown"

selected_strategy = extract_strategy(meta_response)

print("=" * 70)
print("STRATEGY ANALYSIS")
print("=" * 70)
print(f"\nSelected Strategy: {selected_strategy}\n")

# Check if it's a novel strategy
if selected_strategy not in ['CoT', 'ToT', 'AoT', 'Chain of Thought', 
                              'Tree of Thought', 'Algorithm of Thought']:
    print("✨ Meta Thinker INVENTED a new reasoning strategy!")
    print(f"   Novel strategy name: {selected_strategy}\n")
else:
    print("✓ Meta Thinker selected a known strategy")
    print(f"   Strategy: {selected_strategy}\n")

print("=" * 70)

## 12. Test on Different Problem Types

Let's test how well meta-learning generalizes to different reasoning domains.

In [ ]:
# Define problems from different domains
test_problems = {
    'math': """A train travels 120 miles in 2 hours. 
    If it increases its speed by 20 mph, how long will it take to travel 180 miles?""",
    
    'commonsense': """Why do we see lightning before we hear thunder, 
    even though they happen at the same time?""",
    
    'logic': """All birds can fly. Penguins are birds. 
    Therefore, penguins can fly. What's wrong with this argument?"""
}

# Test meta-learner on each problem
print("Testing Meta Thinker on different problem types...\n")

for problem_type, problem in test_problems.items():
    print(f"\nTesting on: {problem_type.upper()}")
    print("=" * 70)
    
    # Create messages with few-shot examples
    messages = few_shot_messages.copy()
    messages.append({'role': 'user', 'content': problem})
    
    # Get response
    response = pipeline_message(messages, client)
    
    # Extract strategy
    strategy = extract_strategy(response)
    
    print(f"Problem: {problem}")
    print(f"\nSelected Strategy: {strategy}")
    print(f"\nResponse Preview: {response[:300]}...")
    print("=" * 70)

## Summary and Key Findings

### What We Demonstrated:

1. **Meta-Reasoning System**: Created a prompt that enables models to select and invent reasoning strategies

2. **Few-Shot Learning**: Showed how providing examples from multiple domains improves strategy selection

3. **Adaptive Strategy Selection**: Meta Thinker chooses appropriate strategies based on problem characteristics

4. **Novel Strategy Invention**: When existing strategies fail, the system can create new reasoning approaches

5. **Cross-Domain Generalization**: The same meta-learning system works across math, logic, and commonsense problems

### Key Insights:

- **Traditional approaches** (fixed CoT/ToT/AoT) work well for standard problems
- **Meta Thinker** excels at:
  - Novel or ambiguous problems
  - Problems requiring hybrid reasoning
  - Tasks where the optimal strategy isn't obvious
  
### Next Steps:

- **Notebook 02**: Deep dive into Game24 reasoning experiments
- **Notebook 03**: Comprehensive testing and evaluation
- Consider fine-tuning models specifically for meta-reasoning

### Related Files:

- `meta_learn.py`: Production script for meta-learning experiments
- `main.py`: Batch processing for multiple datasets